# ClimateVision — Train on Real Sentinel-2 Data

**Runtime:** GPU (T4 recommended) — Runtime → Change runtime type → T4 GPU

**Steps:**
1. Install dependencies
2. Clone the repo
3. Upload your GEE service account key
4. Download real Sentinel-2 training patches from GEE
5. Train the Attention U-Net
6. Download the trained model checkpoint

## 0. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## 1. Clone the repo

In [ ]:
import os

if not os.path.exists('ClimateVision'):
    !git clone https://github.com/Climate-Vision/ClimateVision.git

%cd ClimateVision
!git log --oneline -5

## 2. Install dependencies

In [ ]:
!pip install -q earthengine-api rasterio pillow tqdm pyyaml
!pip install -q -e .
print('Dependencies installed')

## 3. Upload GEE service account key

Upload the file `kinos-473422-be4970a2dee9.json` from your Mac when prompted.

In [ ]:
from google.colab import files
import json, os

print('Upload your GEE service account key JSON file...')
uploaded = files.upload()

key_filename = list(uploaded.keys())[0]
os.makedirs('secrets', exist_ok=True)
os.rename(key_filename, 'secrets/gee-service-account.json')

with open('secrets/gee-service-account.json') as f:
    key_data = json.load(f)

SERVICE_ACCOUNT = key_data['client_email']
PROJECT_ID      = key_data['project_id']
print(f'Service account: {SERVICE_ACCOUNT}')
print(f'Project:         {PROJECT_ID}')

## 4. Authenticate GEE

In [ ]:
import ee

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, 'secrets/gee-service-account.json')
ee.Initialize(credentials)

# Quick test
point = ee.Geometry.Point([-62.0, -3.0])
count = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(point)
           .filterDate('2023-01-01', '2023-12-31')
           .size().getInfo())
print(f'GEE connected! Found {count} Sentinel-2 images over Amazon test point.')

## 5. Download real training data

Downloads Sentinel-2 (R/G/B/NIR) + Google Dynamic World forest labels for 3 regions.
~1500 patches total, 256×256 pixels each at 10m resolution.

In [ ]:
import subprocess, sys

REGIONS = [
    # (label, west, south, east, north, patches)
    ('amazon',  -65.0, -5.0,  -60.0, -1.0,  600),
    ('congo',    22.0, -2.0,   27.0,  2.0,  500),
    ('borneo',  110.0, -2.0,  115.0,  2.0,  400),
]

for label, w, s, e, n, patches in REGIONS:
    print(f'\nDownloading {label} ({patches} patches)...')
    result = subprocess.run([
        sys.executable, 'scripts/prepare_data.py',
        '--mode', 'gee',
        '--bbox', str(w), str(s), str(e), str(n),
        '--start', '2022-01-01',
        '--end',   '2023-12-31',
        '--max-patches', str(patches),
        '--out', 'data/processed',
        '--cloud-threshold', '0.15',
    ], capture_output=True, text=True)
    print(result.stdout[-2000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])

# Count patches
import glob
train_count = len(glob.glob('data/processed/train/images/*.tif'))
val_count   = len(glob.glob('data/processed/val/images/*.tif'))
test_count  = len(glob.glob('data/processed/test/images/*.tif'))
print(f'\nDataset ready: train={train_count}  val={val_count}  test={test_count}')

## 6. Fit normalizer on training set

In [ ]:
result = subprocess.run([
    sys.executable, 'scripts/prepare_data.py',
    '--mode', 'synthetic',   # dummy mode — only --fit-normalizer matters
    '--n-patches', '0',
    '--out', 'data/processed',
    '--fit-normalizer',
    '--normalizer-out', 'data/processed/normalizer.json',
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 7. Train the model

In [ ]:
os.environ['GEE_PROJECT_ID']           = PROJECT_ID
os.environ['GEE_SERVICE_ACCOUNT']      = SERVICE_ACCOUNT
os.environ['GEE_SERVICE_ACCOUNT_KEY']  = 'secrets/gee-service-account.json'

!python scripts/train.py \
    --data-dir data/processed \
    --epochs 50 \
    --batch-size 16 \
    --num-workers 2 \
    --run-name gee_real_data \
    --arch attention_unet

## 8. Evaluate on test set

In [ ]:
import glob
checkpoints = sorted(glob.glob('models/gee_real_data/best_model.pth'))
if checkpoints:
    checkpoint = checkpoints[0]
    print(f'Best checkpoint: {checkpoint}')
    !python scripts/evaluate.py \
        --checkpoint {checkpoint} \
        --data-dir data/processed
else:
    print('No checkpoint found — check training output above')

## 9. Download the trained model

This will download `best_model.pth` to your Mac.
Put it in `ClimateVision-main/models/` and the API will automatically use it.

In [ ]:
from google.colab import files
import shutil

# Copy to root for easy download
shutil.copy('models/gee_real_data/best_model.pth', 'best_model_gee.pth')
files.download('best_model_gee.pth')
print('Download started — save to ClimateVision-main/models/best_model.pth on your Mac')